<a href="https://colab.research.google.com/github/Rasha27-jcu/data-2000/blob/sp26/homework/040_linear-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ucimlrepo

# Homework Assignment: Predicting Heart Disease with Logistic Regression

## Overview

In this lab you will work with a real clinical dataset collected from patients undergoing cardiac evaluation at four medical institutions in the 1980s. Your goal is to build, evaluate, and interpret a binary logistic regression model that predicts whether a patient has heart disease based on a set of diagnostic measurements. Along the way you will practice the full modeling workflow: loading and inspecting raw data, cleaning and encoding features, fitting a logistic regression, and interpreting your results in clinically meaningful terms.

---

## The Dataset

The UCI Heart Disease dataset is one of the most widely studied datasets in machine learning, and for good reason — it is small enough to be tractable, rich enough to be interesting, and grounded in real clinical practice. The data were originally collected across four sites: the Cleveland Clinic Foundation, the Hungarian Institute of Cardiology in Budapest, the University Hospital in Zurich, and the University Hospital in Long Beach. The version most commonly used in practice, and the one you will work with here, comes from Cleveland and contains **303 patient records**.

Each row represents a single patient. The original dataset contains a target variable (`num`) ranging from 0 to 4, indicating the degree of coronary artery narrowing observed during angiography. For this assignment you will **binarize the target**: patients with a value of 0 will be labeled **0 (no heart disease)** and patients with values 1 through 4 will be labeled **1 (heart disease present)**. This reflects the clinically relevant question: does this patient have meaningful coronary artery disease, or not?

---

## Features

The dataset contains 13 predictor variables. Read these descriptions carefully — understanding what each variable actually measures will help you make better decisions during cleaning and will make your interpretation more meaningful.

**`age`** — Age of the patient in years. A continuous variable. Older age is generally associated with higher cardiovascular risk, though the relationship is not perfectly linear.

**`sex`** — Biological sex, encoded as 1 (male) and 0 (female). In the original study population, male patients had higher rates of disease. You should treat this as a binary categorical feature.

**`cp`** — Chest pain type. This is a categorical variable with four values: 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic. Typical angina is the classic presentation of cardiac chest pain — pressure or tightness brought on by exertion and relieved by rest. Importantly, asymptomatic patients (value 4) actually show *higher* rates of disease in this dataset, which is a good reminder not to assume ordinal relationships in categorical variables. You should one-hot encode this feature.

**`trestbps`** — Resting blood pressure in mmHg, measured at the time of hospital admission. Normal resting blood pressure is around 120/80 mmHg; elevated values may indicate hypertension and increased cardiac strain. This is a continuous feature and may contain a small number of physiologically implausible outliers worth inspecting.

**`chol`** — Serum cholesterol in mg/dL. Higher cholesterol, particularly LDL cholesterol, is a well-established risk factor for coronary artery disease. This is continuous. Be aware that a small number of records contain a value of 0, which is physiologically impossible and should be treated as missing.

**`fbs`** — Fasting blood sugar, recorded as a binary indicator: 1 if fasting blood sugar is greater than 120 mg/dL, 0 otherwise. Elevated fasting blood sugar is a marker of diabetes or pre-diabetes, both of which significantly increase cardiovascular risk. Treat this as binary categorical.

**`restecg`** — Resting electrocardiographic results. Another categorical variable with three values: 0 = normal, 1 = ST-T wave abnormality (which can indicate ischemia or other cardiac stress), 2 = left ventricular hypertrophy by Estes' criteria. You should one-hot encode this feature, though be aware that value 2 is rare in this dataset and its coefficient may be unstable.

**`thalach`** — Maximum heart rate achieved during exercise stress testing. This is the peak heart rate recorded during a supervised treadmill or cycling test. Higher maximum heart rate is generally associated with better cardiovascular fitness and lower disease risk, so you may find this feature has a negative relationship with the outcome. This is a continuous variable.

**`exang`** — Exercise-induced angina, encoded as 1 (yes) or 0 (no). This indicates whether the patient experienced chest pain during the exercise stress test. It is one of the most clinically informative features in the dataset, as angina triggered by exertion is a strong indicator of obstructed coronary blood flow. Treat this as binary categorical.

**`oldpeak`** — ST depression induced by exercise relative to rest, measured in millivolts. During an ECG stress test, depression of the ST segment is a sign that part of the heart muscle is not receiving adequate blood flow. A value of 0 means no depression was observed; higher values indicate greater depression and more severe ischemia. This is a continuous feature with a right-skewed distribution and a notable mass of values at exactly 0.

**`slope`** — The slope of the peak exercise ST segment. This describes the shape of the ST segment at maximum exercise: 1 = upsloping, 2 = flat, 3 = downsloping. A downsloping or flat ST segment during peak exercise is considered more clinically concerning than an upsloping one. This should be treated as categorical and one-hot encoded.

**`ca`** — Number of major coronary vessels colored by fluoroscopy, ranging from 0 to 3. Fluoroscopy is an imaging technique used to visualize blood flow through the coronary arteries; a vessel that "colors" well is unobstructed. More obstructed vessels indicates more widespread disease. This is recorded as a numeric variable but functions more like an ordinal count. **It contains missing values coded as `?` in the raw file**, which you will need to handle.

**`thal`** — Results of a thallium stress test, a nuclear imaging procedure that reveals areas of the heart with reduced blood flow. Values are: 3 = normal, 6 = fixed defect (an area that never receives adequate blood flow, often indicating prior heart attack), 7 = reversible defect (an area with reduced flow during stress that recovers at rest, indicating ischemia). This is categorical and should be one-hot encoded. Like `ca`, **it contains missing values coded as `?`** in the raw file.

---

## Your Tasks

1. **Load and inspect** the raw data. Assign column names, identify data types, and produce a summary of missing values and basic descriptive statistics for each feature.

2. **Clean the data.** Address the missing value issues described above. Justify your imputation strategy — mean, median, mode, or drop — for each affected feature, with a brief explanation of why that choice makes sense given the variable's distribution and clinical meaning.

3. **Engineer your features.** Binarize the target variable. One-hot encode the appropriate categorical features. Decide how to handle the binary categoricals and document your reasoning.

4. **Fit a logistic regression model** using a train-test split. Report accuracy, a confusion matrix, and other relevant metrics for interpreting model performance on the test set. Given the clinical context, reflect briefly on whether accuracy is the right metric here, or whether another metric deserves more weight.

In [2]:
import pprint
from ucimlrepo import fetch_ucirepo

In [3]:
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets

In [4]:
# pprint.pprint(heart_disease.metadata, indent=4)
# pprint.pprint(heart_disease.variables, indent=4)

In [6]:
!pip install ucimlrepo
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets

print(X.info())
print(X.describe())
print(X.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB
None
              age         sex          cp    trestbps        chol         fbs  \
count  303.000000  303.000000  303.000000  303.000000  303.000000  303.000000   
mean    54.438944    0.679868    3.158416  131.689769  246.693069    0.148515   
std      9

In [8]:
X = X.replace('?', np.nan).astype(float)

X['ca'] = X['ca'].fillna(X['ca'].median())
X['chol'] = X['chol'].replace(0, X['chol'].median())

X['thal'] = X['thal'].fillna(X['thal'].mode()[0])

print("Data cleaning completed.")

Data cleaning completed.


In [9]:
y_binary = (y > 0).astype(int)

categorical_cols = ['cp', 'restecg', 'slope', 'thal']
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

print(f"Total features after encoding: {X_encoded.shape[1]}")

Total features after encoding: 18


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y_binary, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train.values.ravel())

y_pred = model.predict(X_test)

print("--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

--- Model Performance ---
Accuracy: 0.84

Confusion Matrix:
[[23  6]
 [ 4 28]]

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.79      0.82        29
           1       0.82      0.88      0.85        32

    accuracy                           0.84        61
   macro avg       0.84      0.83      0.83        61
weighted avg       0.84      0.84      0.84        61

